# PyCAM-SIMA complete-CAM interactive workflow

This is the single maintained PyCAM-SIMA Notebook. It controls the complete 24-rank CAM-SIMA model and exposes three Python interfaces:

- `model.options`: explicit settings fixed before `cam_init`;
- `model.step_plan`: the editable top-level CAM phase order used by `model.step()`;
- `model.parameters`: typed handles for reading and writing live CAM fields.

Choose the FKESSLER configuration for Kessler physics or FADIAB for a real SE dynamics-only run. Complete-CAM physics is selected during initialization; it is not disabled by silently skipping `cam_run2`.

## 1. Explicit configuration

Change `physics_profile` before running this cell. Runtime options may be edited until `model.start()` calls `cam_init`; changing them afterward requires a new session and fresh run directory.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import pycam_sima
from pycam_sima import (
    FullCAMRuntimeOptions,
    FullCAMStepPlan,
    NotebookSession,
)
from pycam_sima.config import CaseConfig

repo = Path("/glade/work/ruitong/pycam-sima")
scratch = Path(os.environ.get("SCRATCH", "/glade/derecho/scratch/ruitong"))
physics_profile = "kessler"  # Use "adiabatic" for real dynamics only.

profiles = {
    "kessler": {
        "config": repo / "configs/fkessler_ne3pg3.yaml",
        "case": repo / "reference/cases/FKESSLER_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FKESSLER_ne3pg3_gnu_24x50/FKESSLER_ne3pg3_gnu_24x50/run",
    },
    "adiabatic": {
        "config": repo / "configs/adiabatic_ne3pg3.yaml",
        "case": repo / "reference/cases/FADIAB_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FADIAB_ne3pg3_gnu_24x50/FADIAB_ne3pg3_gnu_24x50/run",
    },
}
selected = profiles[physics_profile]
config = CaseConfig.from_yaml(selected["config"])

options = FullCAMRuntimeOptions(
    timestep_seconds=1800,
    physics_profile=physics_profile,
    mediator_present=False,
)
step_plan = FullCAMStepPlan.default()

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = scratch / "pycam-sima/notebook_trials" / f"{physics_profile}-{stamp}" / "run"
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(selected["reference_run"] / "atm_in", run_dir / "atm_in")

print("pycam_sima", pycam_sima.__version__)
print("run directory:", run_dir)
print("options:", options.describe())
step_plan.describe()

## 2. Start the complete MPI model

From a Derecho login-node kernel, `start()` submits a 24-rank PBS worker. Inside an allocation it launches locally. The cell returns only when all ranks finish initialization and wait for Python commands.

In [ ]:
if "model" in globals() and model.running:
    model.close()

model = NotebookSession(
    config,
    run_dir=run_dir,
    env_script=selected["case"] / ".env_mach_specific.sh",
    python_executable=repo / ".venv/bin/python",
    log_path=run_dir / "mpi-worker.log",
    options=options,
    step_plan=step_plan,
)
model.start()
print(
    f"ready: mode={model.launch_mode_used}, job={model.job_id}, "
    f"ranks={model.ranks}, fields={len(model.field_names)}, step={model.current_step}"
)

## 3. Inspect the Python control interfaces

The plan shown here is the actual plan sent to every MPI rank by `model.step()`. `parameters.describe()` lists the typed key fields and every additional field available through `parameters.field(name)`.

In [ ]:
print("runtime options:", model.options.describe())
print("phase status:", model.phase_status)
print("step plan:")
display(model.step_plan.describe())
print("parameters and fields:")
model.parameters.describe()

## 4. Read a live CAM field

The typed field handle makes the parameter name explicit. `get()` transfers a rank-local NumPy copy to the Notebook; `stats()` computes a compact summary on the MPI worker.

In [ ]:
temperature_field = model.parameters.air_temperature
print(temperature_field.info)
print(temperature_field.stats(rank=0))
temperature = temperature_field.get(rank=0)
temperature

## 5. Advance one complete timestep

`step()` executes `model.step_plan` collectively on all 24 ranks, then all ranks return to the command wait loop. The default plan preserves the validated CAM order.

In [ ]:
step = model.step()
print("completed step:", step)
print(temperature_field.stats(rank=0))

## 6. Pause after each top-level CAM phase

`run_phase()` sends exactly one call to every MPI rank. Re-run the next cell to walk through `cam_run2`, SE dynamics, finalization, clock advancement, initialization, and `cam_run1`.

In [ ]:
phase = model.next_phase
status = model.run_phase(phase)
stats = temperature_field.stats(rank=0)
print(
    f"finished={phase} next={status['next_phase']} "
    f"step={status['step']} native_nstep={status['native_nstep']} "
    f"Tmean={stats['mean']:.17g}"
)

## 7. Optional targeted field modification

Field edits are allowed at every Python boundary. `set()` writes the supplied values into CAM memory on the selected rank. This intentionally breaks BFB.

In [ ]:
# changed = temperature_field.get(rank=0)
# changed[0, 0] += 0.01
# temperature_field.set(changed, rank=0)
# print(temperature_field.stats(rank=0))

## 8. Optional process on/off and ordering experiments

Every complete-CAM phase is required by the validated sequence. Disabling SE dynamics or changing order therefore requires `unsafe=True`. The modified plan is used by the next `model.step()` and may fail inside CAM if the requested sequence violates native lifecycle assumptions. After an unsafe step, restart CAM before returning to the default plan.

For a scientifically valid dynamics-only run, set `physics_profile = "adiabatic"` in section 1 and start a fresh FADIAB session.

In [ ]:
# Turn off the real SE dynamics call for an explicit control experiment:
# model.step_plan.disable("cam_run3", unsafe=True)

# Or change the complete-CAM order:
# model.step_plan.move("cam_run3", before="cam_run2", unsafe=True)

# model.step()
model.step_plan.describe()

## 9. Optional complete 50-step run

Run this only with the unmodified default plan and no field edits if the goal is BFB validation.

In [ ]:
# while model.current_step < config.steps:
#     model.step()
#     print(model.current_step, temperature_field.stats(rank=0)["mean"])

## 10. Finalize

Always close the session so CAM finalizes and the MPI worker exits.

In [ ]:
model.close()
print("closed:", run_dir)

## 11. BFB comparison

The comparison is intentionally fail-closed. A partial run reports `bfb=False` with missing files even when every available timestamp matches. A complete 50-step run should contain 51 files.

In [ ]:
from pycam_sima.history_compare import compare_history

comparison = compare_history(selected["reference_run"], run_dir)
comparison.to_dict()